In [ ]:
import os

from detonator import make_db_connection, get_logger
from pandas import DataFrame
from yfinance import Ticker

make_db_connection(db='mongogo-test')

_logger = get_logger('dev')

os.environ['HTTP_PROXY'] = 'socks5://127.0.0.1:8001'
os.environ['HTTPS_PROXY'] = 'socks5://127.0.0.1:8001'



In [ ]:
period = '1mo'
# 1d,5d,1mo,3mo,6mo,1y,2y,5y,10y,ytd,max
interval = '1d'
# 1m,2m,5m,15m,30m,60m,90m,1h,1d,5d,1wk,1mo,3mo
t = Ticker('aapl')
start = None  # YYYY-mm-dd
end = None  # YYYY-mm-dd
his: DataFrame = t.history(period=period, interval=interval, start=start, end=end)
info = t.info

In [ ]:
his.rename(columns={'Open': 'open', 'High': 'high', 'Low': 'low', 'Close': 'close', 'Volume': 'volume',
                    'Dividends': 'dividends',
                    'Stock Splits': 'stock_splits'}, inplace=True)
his.columns

In [ ]:
import pandas as pd

trade_dates = pd.to_datetime(his.index).to_pydatetime()
his['trade_date'] = his.index.strftime('%Y,%m,%d,%H,%M,%S,%f')
print(type(trade_dates[0]))

In [ ]:
his.index.strftime('%Y,%m,%d,%H,%M,%S,%f')

In [ ]:
# import pandas as pd

# pd.to_datetime(his.index.values).to_pydatetime()

In [ ]:
ticker = 'AAPL'
his['ticker'] = ticker
his['interval'] = interval

In [ ]:
his.index.dtype

In [ ]:
his.to_json(orient='records')

In [ ]:
from detonator import df_2_mongo
from dataminer.models import TickerDailyInfo, regulate_ticker_daily_info

make_db_connection(db='mongogo-test')

df_2_mongo(his, TickerDailyInfo)

In [ ]:
regulated_info = regulate_ticker_daily_info(info)
tdi: TickerDailyInfo = TickerDailyInfo.objects.order_by('-trade_date').first()
print(regulated_info)

In [ ]:
print(tdi.to_json())
tdi.update(**regulated_info)
tdi.save()


In [ ]:
from mongoengine import Document, DateTimeField
import numpy as np
import pandas as pd

# Sample NumPy datetime64 array
datetime_array = np.array(['2022-01-01T12:34:56.789012345', '2022-01-02T01:23:45.678901234'], dtype='datetime64[ns]')

# Convert NumPy datetime64 to Python datetime.datetime
python_datetime_array = pd.to_datetime(datetime_array).to_pydatetime()


# Define a MongoDB document with a DateTimeField
class MyDocument(Document):
    timestamp = DateTimeField()


# Connect to the MongoDB database (replace 'your_db' and 'your_host' with your actual database and host)

# Create and save documents with the datetime field
for dt in python_datetime_array:
    doc = MyDocument(timestamp=dt)
    doc.save()


In [ ]:
his.index.values

In [ ]:
datetime_array

In [ ]:


from dataminer.models import TradeCalendar
from dataminer import TradeCalendarShovel
from detonator import make_db_connection

make_db_connection(db='mongogo=test')
tc = TradeCalendarShovel.get_instance()

trade_date = '20231223'
trade_dates = TradeCalendar.objects(cal_date__gt=trade_date, cal_date__lte=tc.last_us_trade_day_before_today(),
                                    country='us', is_open=True)


In [ ]:
[t.cal_date for t in trade_dates]

In [ ]:
aaaa = ['a']
aaaa[-1]